# 1. Diseño del esquema que almacenará los datos

## Descripción del conjunto de datos

Para el desarrollo de esta actividad se seleccionó el dataset **FIDE World Chess Ratings (201K Players)** obtenido desde Kaggle. Este conjunto de datos contiene información de más de 200.000 jugadores registrados por la Federación Internacional de Ajedrez (FIDE), incluyendo identificadores únicos, federación representada, género, títulos oficiales y rating Elo.

Cada registro representa un jugador y sus características competitivas dentro del sistema de clasificación FIDE.

## Diccionario de datos

| Campo  | Tipo de dato | Nulabilidad | Descripción                                 |
| ------ | ------------ | ----------- | ------------------------------------------- |
| id     | BIGINT       | No          | Identificador único del jugador             |
| name   | STRING       | No          | Nombre completo del jugador                 |
| fed    | STRING       | Sí          | Federación o país representado              |
| sex    | STRING       | Sí          | Género del jugador                          |
| title  | STRING       | Sí          | Título FIDE principal                       |
| wtitle | STRING       | Sí          | Título femenino FIDE                        |
| otitle | STRING       | Sí          | Otros títulos reconocidos                   |
| foa    | STRING       | Sí          | Información adicional de afiliación         |
| rating | INT          | No          | Rating Elo del jugador                      |
| games  | INT          | No          | Número de partidas registradas              |
| k      | INT          | No          | Factor K utilizado en el cálculo del rating |
| bday   | INT          | Sí          | Año de nacimiento                           |

## Llaves y restricciones

* Llave primaria: id.
* El campo name es obligatorio para identificar al jugador.
* Los campos rating, games y k almacenan valores numéricos enteros.
* Los campos relacionados con títulos permiten valores nulos debido a que no todos los jugadores poseen títulos oficiales.
* El campo bday puede contener valores nulos cuando la información de nacimiento no está disponible.

## Justificación del diseño

El esquema se diseñó utilizando una única entidad denominada FIDE_PLAYER debido a que cada fila representa un jugador independiente. Esta estructura simplifica el análisis de información relacionada con ratings, federaciones, títulos y actividad competitiva, permitiendo realizar consultas analíticas eficientes mediante Spark SQL.


##Diagrama Mermaid

![image_1780766948949.png](./image_1780766948949.png "image_1780766948949.png")

erDiagram

    FIDE_PLAYER {
        BIGINT id PK
        STRING name
        STRING fed
        STRING sex
        STRING title
        STRING wtitle
        STRING otitle
        STRING foa
        INT rating
        INT games
        INT k
        INT bday
    }

##StructType PySpark

In [0]:
from pyspark.sql.types import *

fide_schema = StructType([
    StructField("id", LongType(), False),
    StructField("name", StringType(), False),
    StructField("fed", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("title", StringType(), True),
    StructField("wtitle", StringType(), True),
    StructField("otitle", StringType(), True),
    StructField("foa", StringType(), True),
    StructField("rating", IntegerType(), True),
    StructField("games", IntegerType(), True),
    StructField("k", IntegerType(), True),
    StructField("bday", IntegerType(), True)
])

fide_schema

StructType([StructField('id', LongType(), False), StructField('name', StringType(), False), StructField('fed', StringType(), True), StructField('sex', StringType(), True), StructField('title', StringType(), True), StructField('wtitle', StringType(), True), StructField('otitle', StringType(), True), StructField('foa', StringType(), True), StructField('rating', IntegerType(), True), StructField('games', IntegerType(), True), StructField('k', IntegerType(), True), StructField('bday', IntegerType(), True)])

# Configuración de la infraestructura

La actividad fue desarrollada utilizando Databricks Free Edition con cómputo Serverless administrado por Databricks.

La plataforma proporciona un entorno de procesamiento distribuido basado en Apache Spark, permitiendo ejecutar tareas de análisis de datos sin necesidad de administrar infraestructura física o virtual.

Características principales:

- Plataforma: Databricks Free Edition
- Tipo de cómputo: Serverless
- Motor de procesamiento: Apache Spark
- Lenguaje principal: Python
- Almacenamiento: DBFS y Unity Catalog
- Sistema de consultas: Spark SQL

In [0]:
import sys

print("Versión de Python:")
print(sys.version)

print("\nVersión de Spark:")
print(spark.version)

Versión de Python:
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

Versión de Spark:
4.1.0


In [0]:
conf = spark.conf.getAll

for clave, valor in conf.items():
    print(f"{clave}: {valor}")

spark.databricks.execution.timeout: 9000
spark.sql.ansi.enabled: true
spark.sql.shuffle.partitions: auto


In [0]:
display(dbutils.fs.ls("/"))

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


### Evidencias recopiladas

- Versión de Python: 3.12.3
- Versión de Apache Spark: 4.1.0
- Configuración activa del entorno Spark.
- Estructura de almacenamiento utilizada para el proyecto.

# 3. Obtención de datos de Kaggle y creación de la tabla

## Obtención del dataset

Para esta actividad se seleccionó el conjunto de datos **FIDE World Chess Ratings (201K Players)** disponible en Kaggle. Debido a la facilidad de uso en Databricks Free Edition, se utilizó la opción de descarga manual del archivo CSV y posterior carga al entorno mediante un Volume administrado por Databricks.

### Ruta del archivo cargado

dbfs:/Volumes/workspace/default/fideworldchessratings201kplayers/03_FIDE_Chess_Ratings.csv

El uso de Volumes permite almacenar y acceder a los archivos de manera organizada dentro del entorno de Databricks, facilitando su lectura mediante Apache Spark.

---

## Lectura de los datos con Spark

Una vez cargado el archivo, se procedió a leer el conjunto de datos utilizando Spark DataFrame.

Se configuró la lectura con encabezados y detección automática de tipos de datos.


In [0]:
df = spark.read.csv(
    "dbfs:/Volumes/workspace/default/fideworldchessratings201kplayers/03_FIDE_Chess_Ratings.csv",
    header=True,
    inferSchema=True
)

display(df)

id,name,fed,sex,title,wtitle,otitle,foa,rating,games,k,bday
53707043,A Darshil,IND,M,NA,NA,NA,NA,1412,4,40,2013
53200465,"A F M Ehteshamul, Hoque (tuhin",BAN,M,NA,NA,NA,NA,1797,0,40,1977
5716365,"A Hamid, Harman",MAS,M,NA,NA,NA,NA,1552,0,20,1970
53200553,A I Sabbir,BAN,M,NA,NA,NA,NA,1607,0,40,1995
5045886,"A K, Kalshyan",IND,M,NA,NA,NA,NA,1747,0,20,1964
10291695,"A S M Mashrur, Zaman",BAN,M,NA,NA,NA,NA,1614,0,40,2008
10257853,"A, L M Wayes Faruki",BAN,M,NA,NA,NA,NA,1765,0,40,2015
33454710,A. HUDSON,IND,M,NA,NA,NA,AFM,1644,0,40,2013
7109229,Aa Djoni Kosanda,INA,M,NA,NA,NA,NA,1691,0,20,1953
1701991,"Aaberg, Anton",SWE,M,IM,NA,NA,NA,2328,0,10,1972


## Validación del esquema

Para verificar que Spark interpretó correctamente la estructura del archivo, se inspeccionó el esquema del DataFrame.

In [0]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- fed: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- title: string (nullable = true)
 |-- wtitle: string (nullable = true)
 |-- otitle: string (nullable = true)
 |-- foa: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- games: integer (nullable = true)
 |-- k: integer (nullable = true)
 |-- bday: integer (nullable = true)



## Exploración inicial del conjunto de datos

Se realizó una validación básica para conocer el tamaño del dataset cargado.

In [0]:
print("Número de registros:", df.count())
print("Número de columnas:", len(df.columns))

Número de registros: 201015
Número de columnas: 12


In [0]:
print(df.columns)

['id', 'name', 'fed', 'sex', 'title', 'wtitle', 'otitle', 'foa', 'rating', 'games', 'k', 'bday']


## Estadísticas descriptivas

Para obtener una visión general de las variables numéricas se generó un resumen estadístico.


In [0]:
display(df.describe())

summary,id,name,fed,sex,title,wtitle,otitle,foa,rating,games,k,bday
count,201015,201015,201015,201015,201015,201015,201015,201015,201015,201015,201015,201015
mean,2.2859665311021566E7,null,null,null,null,null,null,null,1744.6333109469442,1.7010770340521852,30.733825833892993,1990.8368032236401
stddev,2.2951918040279794E7,null,null,null,null,null,null,null,224.9957568873682,3.788444977205018,10.270465940263739,20.77963079208988
min,100013,A Darshil,AFG,F,CM,NA,AO,ACM,1400,0,10,1900
max,94799962,"zur Megede, Detlef, Dr.",ZIM,M,WIM,WIM,SLI,NA,2839,45,40,2021


## Creación de la tabla administrada

Después de validar la información cargada, se creó una tabla administrada denominada `fide_players`, la cual será utilizada para las consultas analíticas posteriores.


In [0]:
df.write.mode("overwrite").saveAsTable("fide_players")

## Verificación de la tabla creada

Se comprobó que la tabla fue registrada correctamente en el catálogo de Spark SQL.


In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|default |fide_players|false      |
+--------+------------+-----------+



## Descripción de la tabla

Para validar la estructura almacenada se consultaron los metadatos de la tabla creada.


In [0]:
%sql
DESCRIBE TABLE fide_players;

col_name,data_type,comment
id,int,null
name,string,null
fed,string,null
sex,string,null
title,string,null
wtitle,string,null
otitle,string,null
foa,string,null
rating,int,null
games,int,null


## Consulta de validación

Finalmente, se verificó el contenido almacenado mediante una consulta simple.

In [0]:
%sql
SELECT *
FROM fide_players
LIMIT 10;

id,name,fed,sex,title,wtitle,otitle,foa,rating,games,k,bday
53707043,A Darshil,IND,M,NA,NA,NA,NA,1412,4,40,2013
53200465,"A F M Ehteshamul, Hoque (tuhin",BAN,M,NA,NA,NA,NA,1797,0,40,1977
5716365,"A Hamid, Harman",MAS,M,NA,NA,NA,NA,1552,0,20,1970
53200553,A I Sabbir,BAN,M,NA,NA,NA,NA,1607,0,40,1995
5045886,"A K, Kalshyan",IND,M,NA,NA,NA,NA,1747,0,20,1964
10291695,"A S M Mashrur, Zaman",BAN,M,NA,NA,NA,NA,1614,0,40,2008
10257853,"A, L M Wayes Faruki",BAN,M,NA,NA,NA,NA,1765,0,40,2015
33454710,A. HUDSON,IND,M,NA,NA,NA,AFM,1644,0,40,2013
7109229,Aa Djoni Kosanda,INA,M,NA,NA,NA,NA,1691,0,20,1953
1701991,"Aaberg, Anton",SWE,M,IM,NA,NA,NA,2328,0,10,1972


## Conclusión

El conjunto de datos fue descargado desde Kaggle y cargado exitosamente en Databricks mediante un Volume administrado. Posteriormente se realizó la lectura con Apache Spark, la validación de su estructura y la creación de una tabla administrada denominada `fide_players`. Esta tabla quedó disponible para realizar consultas analíticas utilizando Spark SQL sobre más de 200.000 registros de jugadores pertenecientes a la Federación Internacional de Ajedrez (FIDE).


# 4. Validaciones en Spark y SQL

## Objetivo

Una vez creada la tabla `fide_players`, se realizaron diferentes validaciones utilizando Apache Spark (PySpark) y Spark SQL. Estas validaciones permiten comprobar que los datos fueron cargados correctamente, verificar la estructura de la tabla y comparar resultados obtenidos mediante ambos enfoques de consulta.

---

# 4.1 Validación de metadatos

## Propósito

Verificar la estructura de la tabla, nombres de columnas y tipos de datos almacenados.

**PySpark**

In [0]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- fed: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- title: string (nullable = true)
 |-- wtitle: string (nullable = true)
 |-- otitle: string (nullable = true)
 |-- foa: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- games: integer (nullable = true)
 |-- k: integer (nullable = true)
 |-- bday: integer (nullable = true)



**Spark SQL**

In [0]:
%sql
DESCRIBE TABLE fide_players;

col_name,data_type,comment
id,int,null
name,string,null
fed,string,null
sex,string,null
title,string,null
wtitle,string,null
otitle,string,null
foa,string,null
rating,int,null
games,int,null


**Spark SQL (definición de la tabla)**

In [0]:
%sql
SHOW CREATE TABLE fide_players;

createtab_stmt
"CREATE TABLE workspace.default.fide_players ( id INT, name STRING COLLATE UTF8_BINARY, fed STRING COLLATE UTF8_BINARY, sex STRING COLLATE UTF8_BINARY, title STRING COLLATE UTF8_BINARY, wtitle STRING COLLATE UTF8_BINARY, otitle STRING COLLATE UTF8_BINARY, foa STRING COLLATE UTF8_BINARY, rating INT, games INT, k INT, bday INT) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


### Interpretación

Los resultados muestran la estructura de la tabla, los nombres de las columnas y los tipos de datos utilizados para almacenar la información de los jugadores FIDE. Esta validación permite confirmar que el esquema cargado coincide con el diseño propuesto anteriormente.

# 4.2 Descripción de los datos

## Propósito

Obtener estadísticas descriptivas básicas para las variables numéricas del conjunto de datos.


**PySpark**

In [0]:
df.describe().show()

+-------+--------------------+--------------------+------+------+------+------+------+------+------------------+------------------+------------------+------------------+
|summary|                  id|                name|   fed|   sex| title|wtitle|otitle|   foa|            rating|             games|                 k|              bday|
+-------+--------------------+--------------------+------+------+------+------+------+------+------------------+------------------+------------------+------------------+
|  count|              201015|              201015|201015|201015|201015|201015|201015|201015|            201015|            201015|            201015|            201015|
|   mean|2.2859665311021566E7|                NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL|1744.6333109469442|1.7010770340521852|30.733825833892993|1990.8368032236401|
| stddev|2.2951918040279794E7|                NULL|  NULL|  NULL|  NULL|  NULL|  NULL|  NULL| 224.9957568873682| 3.788444977205018|10.270465940263739|

**Spark SQL**

In [0]:
%sql
SELECT
    COUNT(*) AS total_jugadores,
    MIN(rating) AS rating_minimo,
    MAX(rating) AS rating_maximo,
    ROUND(AVG(rating),2) AS rating_promedio
FROM fide_players;

total_jugadores,rating_minimo,rating_maximo,rating_promedio
201015,1400,2839,1744.63


### Interpretación

Estas consultas permiten conocer la cantidad total de jugadores registrados y los valores mínimo, máximo y promedio del rating Elo, proporcionando una visión general de la distribución de los datos.

# 4.3 Conteo de registros

## Propósito

Validar que la cantidad de registros almacenados en la tabla coincide con la cantidad cargada desde el archivo CSV.

**PySpark**

In [0]:
print("Total de registros:", df.count())

Total de registros: 201015


**Spark SQL**

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM fide_players;

total_registros
201015


### Interpretación

El resultado debe coincidir en ambos casos, confirmando que todos los registros fueron almacenados correctamente.

# 4.4 Muestra de registros

## Propósito

Visualizar ejemplos de los datos almacenados.

**PySpark**

In [0]:
df.show(10)

+--------+--------------------+---+---+-----+------+------+---+------+-----+---+----+
|      id|                name|fed|sex|title|wtitle|otitle|foa|rating|games|  k|bday|
+--------+--------------------+---+---+-----+------+------+---+------+-----+---+----+
|53707043|           A Darshil|IND|  M|   NA|    NA|    NA| NA|  1412|    4| 40|2013|
|53200465|A F M Ehteshamul,...|BAN|  M|   NA|    NA|    NA| NA|  1797|    0| 40|1977|
| 5716365|     A Hamid, Harman|MAS|  M|   NA|    NA|    NA| NA|  1552|    0| 20|1970|
|53200553|          A I Sabbir|BAN|  M|   NA|    NA|    NA| NA|  1607|    0| 40|1995|
| 5045886|       A K, Kalshyan|IND|  M|   NA|    NA|    NA| NA|  1747|    0| 20|1964|
|10291695|A S M Mashrur, Zaman|BAN|  M|   NA|    NA|    NA| NA|  1614|    0| 40|2008|
|10257853| A, L M Wayes Faruki|BAN|  M|   NA|    NA|    NA| NA|  1765|    0| 40|2015|
|33454710|           A. HUDSON|IND|  M|   NA|    NA|    NA|AFM|  1644|    0| 40|2013|
| 7109229|    Aa Djoni Kosanda|INA|  M|   NA|    NA|  

**Spark SQL**

In [0]:
%sql
SELECT *
FROM fide_players
LIMIT 10;

id,name,fed,sex,title,wtitle,otitle,foa,rating,games,k,bday
53707043,A Darshil,IND,M,NA,NA,NA,NA,1412,4,40,2013
53200465,"A F M Ehteshamul, Hoque (tuhin",BAN,M,NA,NA,NA,NA,1797,0,40,1977
5716365,"A Hamid, Harman",MAS,M,NA,NA,NA,NA,1552,0,20,1970
53200553,A I Sabbir,BAN,M,NA,NA,NA,NA,1607,0,40,1995
5045886,"A K, Kalshyan",IND,M,NA,NA,NA,NA,1747,0,20,1964
10291695,"A S M Mashrur, Zaman",BAN,M,NA,NA,NA,NA,1614,0,40,2008
10257853,"A, L M Wayes Faruki",BAN,M,NA,NA,NA,NA,1765,0,40,2015
33454710,A. HUDSON,IND,M,NA,NA,NA,AFM,1644,0,40,2013
7109229,Aa Djoni Kosanda,INA,M,NA,NA,NA,NA,1691,0,20,1953
1701991,"Aaberg, Anton",SWE,M,IM,NA,NA,NA,2328,0,10,1972


### Interpretación

La muestra permite verificar visualmente que los datos fueron cargados correctamente y que cada columna contiene información coherente.

# 4.5 Consulta SELECT con filtro

## Propósito

Identificar jugadores con un rating elevado.

**PySpark**

In [0]:
df.filter(df.rating > 2500) \
  .select("name","fed","rating") \
  .show(20)

+--------------------+---+------+
|                name|fed|rating|
+--------------------+---+------+
|       Abasov, Nijat|AZE|  2578|
|Abdusattorov, Nod...|UZB|  2771|
|          Acs, Peter|HUN|  2563|
|      Adams, Michael|ENG|  2663|
|         Adhiban, B.|IND|  2534|
|       Aditya Mittal|IND|  2560|
|         Adly, Ahmed|EGY|  2589|
|    Agdestein, Simen|NOR|  2563|
|    Ahmadzada, Ahmad|AZE|  2516|
|   Akobian, Varuzhan|USA|  2552|
|   Akopian, Vladimir|USA|  2582|
|Albornoz Cabrera,...|CUB|  2546|
|  Alekseenko, Kirill|AUT|  2652|
|    Alekseev, Evgeny|ISR|  2523|
|  Alexakis, Dimitris|GRE|  2543|
|Ali Marandi, Cemi...|TUR|  2514|
|Almeida Quintana,...|CUB|  2501|
|Alonso Rosell, Alvar|ESP|  2536|
|         Amar, Elham|NOR|  2584|
|        Amin, Bassem|EGY|  2633|
+--------------------+---+------+
only showing top 20 rows


**Spark SQL**

In [0]:
%sql
SELECT name, fed, rating
FROM fide_players
WHERE rating > 2500
LIMIT 20;

name,fed,rating
"Abasov, Nijat",AZE,2578
"Abdusattorov, Nodirbek",UZB,2771
"Acs, Peter",HUN,2563
"Adams, Michael",ENG,2663
"Adhiban, B.",IND,2534
Aditya Mittal,IND,2560
"Adly, Ahmed",EGY,2589
"Agdestein, Simen",NOR,2563
"Ahmadzada, Ahmad",AZE,2516
"Akobian, Varuzhan",USA,2552


### Interpretación

Esta consulta permite identificar jugadores de alto rendimiento competitivo, normalmente asociados a maestros internacionales y grandes maestros.

# 4.6 Consulta GROUP BY

## Propósito

Analizar la distribución de jugadores por federación.

**PySpark**

In [0]:
from pyspark.sql.functions import count

df.groupBy("fed") \
  .agg(count("*").alias("total_jugadores")) \
  .orderBy("total_jugadores", ascending=False) \
  .show(10)

+---+---------------+
|fed|total_jugadores|
+---+---------------+
|ESP|          17858|
|IND|          16445|
|FRA|          15958|
|GER|          13310|
|RUS|           7228|
|ITA|           6702|
|POL|           5946|
|CZE|           5557|
|USA|           4495|
|NED|           4227|
+---+---------------+
only showing top 10 rows


**Spark SQL**

In [0]:
%sql
SELECT
    fed,
    COUNT(*) AS total_jugadores
FROM fide_players
GROUP BY fed
ORDER BY total_jugadores DESC
LIMIT 10;

fed,total_jugadores
ESP,17858
IND,16445
FRA,15958
GER,13310
RUS,7228
ITA,6702
POL,5946
CZE,5557
USA,4495
NED,4227


### Interpretación

El resultado muestra las federaciones con mayor cantidad de jugadores registrados en el ranking FIDE.


# 4.7 Consulta GROUP BY sobre títulos FIDE

## Propósito

Identificar la distribución de títulos oficiales entre los jugadores.

**PySpark**

In [0]:
df.groupBy("title") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+-----+------+
|title| count|
+-----+------+
|   NA|188366|
|   FM|  4762|
|   IM|  2433|
|   CM|  2073|
|   GM|  1306|
|  WFM|   825|
|  WCM|   652|
|  WIM|   413|
|  WGM|   185|
+-----+------+



**Spark SQL**

In [0]:
%sql
SELECT
    title,
    COUNT(*) AS cantidad
FROM fide_players
GROUP BY title
ORDER BY cantidad DESC;

title,cantidad
NA,188366
FM,4762
IM,2433
CM,2073
GM,1306
WFM,825
WCM,652
WIM,413
WGM,185


### Interpretación

Esta consulta permite conocer cuántos jugadores poseen títulos como GM (Grandmaster), IM (International Master), FM (FIDE Master) y otros reconocidos por la FIDE.

---

## Conclusión

Las validaciones realizadas mediante PySpark y Spark SQL produjeron resultados consistentes, demostrando que la tabla fue creada correctamente y que los datos pueden analizarse utilizando tanto la API de Spark como consultas SQL tradicionales. Esto confirma la correcta integración entre el almacenamiento, el procesamiento distribuido y las capacidades analíticas ofrecidas por Databricks.


# 5. Ventajas y desventajas: SQL vs Spark (PySpark)

## Introducción

Durante el desarrollo de la actividad se utilizaron dos enfoques para consultar y analizar los datos almacenados en la tabla `fide_players`: Spark SQL y PySpark. Ambos permiten trabajar sobre los mismos datos, pero presentan diferencias importantes en términos de facilidad de uso, flexibilidad y escalabilidad.

## Comparación entre SQL y Spark (PySpark)

| Aspecto          | SQL                                                     | Spark (PySpark)                                                         |
| ---------------- | ------------------------------------------------------- | ----------------------------------------------------------------------- |
| Facilidad de uso | Sintaxis sencilla y fácil de aprender.                  | Requiere conocimientos de Python y Spark.                               |
| Tipo de enfoque  | Declarativo: se indica qué resultado se desea obtener.  | Procedimental: permite definir paso a paso las transformaciones.        |
| Integración      | Excelente integración con herramientas BI y analíticas. | Integración con bibliotecas de análisis avanzado y Machine Learning.    |
| Escalabilidad    | Adecuada para consultas analíticas y reportes.          | Diseñado para procesar grandes volúmenes de datos de forma distribuida. |
| Flexibilidad     | Limitada para procesos complejos o personalizados.      | Permite crear funciones, transformaciones y pipelines avanzados.        |

## Ventajas de SQL

* Sintaxis simple y fácil de interpretar.
* Ideal para consultas exploratorias y análisis descriptivos.
* Permite generar reportes rápidamente.
* Amplia compatibilidad con herramientas de Business Intelligence.
* Facilita la colaboración entre perfiles técnicos y de negocio.

## Desventajas de SQL

* Menor flexibilidad para procesos complejos.
* Limitaciones al implementar lógica avanzada.
* Dependencia de funciones específicas del motor utilizado.
* Menor capacidad para construir pipelines de procesamiento extensos.
* Integración limitada con modelos de Machine Learning.

## Ventajas de Spark (PySpark)

* Procesamiento distribuido y escalable.
* Amplia API para transformación y manipulación de datos.
* Permite crear funciones personalizadas (UDFs).
* Integración con bibliotecas de Machine Learning mediante MLlib.
* Adecuado para procesos ETL y análisis de grandes volúmenes de datos.

## Desventajas de Spark (PySpark)

* Curva de aprendizaje más alta.
* Requiere conocimientos de programación en Python.
* El código puede ser más extenso que una consulta SQL equivalente.
* Algunas operaciones necesitan optimización para obtener buen rendimiento.
* Mayor complejidad en tareas simples.

## Conclusión

A partir de las pruebas realizadas en Databricks, se observó que SQL resulta más adecuado para consultas rápidas, exploración de datos y generación de reportes, mientras que PySpark ofrece mayor flexibilidad y escalabilidad para desarrollar procesos de transformación complejos y análisis avanzados sobre grandes volúmenes de información.

Por esta razón, ambos enfoques son complementarios dentro del ecosistema Databricks. SQL facilita el análisis y la visualización de datos, mientras que PySpark permite construir soluciones de procesamiento y analítica más robustas y escalables.
